# Video Game Market — Exploratory Analysis

**Dataset:** [Video Game Market Price and Revenue Dataset](https://www.kaggle.com/datasets/amith1707/video-game-market-price-and-revenue-dataset) — 5.58M rows, 130 columns, 10,000 simulated games across 6 platforms (2019–2024)  
**Tools:** DuckDB · pandas · matplotlib  
**Database:** Built by `engineering/ingest.py` — run that first if you haven't already

---

## What this notebook does

Answers four business questions about the video game market using SQL queries against a locally built DuckDB database.

| # | Business question |
|---|---|
| 1 | Which genres generate the most average weekly revenue? |
| 2 | How do player counts and revenue change across a game's lifecycle? |
| 3 | Which platform retains the most players and generates the most revenue? |
| 4 | Do discounts actually drive player count increases? |

---

## Database structure (star schema)

The raw 5GB CSV is stored in four tables in a local DuckDB file. This structure avoids repeating static game information (developer, genre, release date) across every weekly row — instead that info lives once in `dim_games` and gets joined in when needed.

| Table | Rows | Contents |
|---|---|---|
| `dim_games` | 9,995 | One row per game — name, genre, developer, release info |
| `dim_platform` | 6 | One row per platform — PC, PS5, PS4, Xbox Series X/S, Xbox One, Switch |
| `dim_time` | 2,190 | One row per date — year, month, quarter, season flags |
| `fact_weekly_metrics` | 5,584,706 | One row per game × platform × week — all measurements |

---

## Setup

Connect to the DuckDB database and confirm all four tables are present with the expected row counts.

**Why DuckDB instead of loading the CSV with pandas?**  
The raw CSV is ~5GB. Pandas would load the entire file into RAM, which is slow and may crash on a laptop. DuckDB queries the data on disk and only pulls what it needs — making it much faster for large analytical queries.

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

# ── Connect to the database ───────────────────────────────────────────────────
# The database file lives in data/ at the project root.
# Path('..') moves one level up from analysis/ to the project root.
DB_PATH = Path('../data/videogames.duckdb')

if not DB_PATH.exists():
    raise FileNotFoundError(
        f'Database not found at {DB_PATH}. '
        'Run engineering/ingest.py first.'
    )

con = duckdb.connect(str(DB_PATH))

# ── Sanity check — confirm all tables loaded correctly ────────────────────────
print('Table row counts:')
for table in ['dim_games', 'dim_platform', 'dim_time', 'fact_weekly_metrics']:
    count = con.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f'  {table:<25} {count:>12,} rows')

---

## Analysis 1 — Genre Revenue

**Business question:** Which game genres generate the most average weekly revenue?

**Why this matters:** For a publisher or investor, knowing which genres command the highest revenue informs where to allocate development budgets. For a platform (Steam, PlayStation Store), it informs which genres to prioritise for featuring and promotion.

**How the query works:**
- We `JOIN fact_weekly_metrics` (which has revenue numbers) to `dim_games` (which has the genre label) using `game_id` as the link between them — this is the star schema in action
- `GROUP BY genre` collapses all weekly observations into one summary row per genre
- `AVG(estimated_revenue_usd)` gives average revenue per weekly observation across all games in that genre
- We also compute the **median** — if the average is much higher than the median, a small number of blockbuster titles are pulling the average up (skewed distribution)

In [ ]:
df_genre = con.execute("""
    SELECT
        g.genre,
        COUNT(DISTINCT f.game_id)                 AS game_count,
        ROUND(AVG(f.estimated_revenue_usd), 2)    AS avg_weekly_revenue,
        ROUND(MEDIAN(f.estimated_revenue_usd), 2) AS median_weekly_revenue,
        ROUND(AVG(f.concurrent_players), 0)       AS avg_players
    FROM fact_weekly_metrics f
    JOIN dim_games g ON f.game_id = g.game_id
    GROUP BY g.genre
    ORDER BY avg_weekly_revenue DESC
    LIMIT 15
""").df()

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(
    df_genre['genre'],
    df_genre['avg_weekly_revenue'] / 1_000_000,
    color='steelblue'
)
ax.set_xlabel('Avg Weekly Revenue (USD millions)')
ax.set_title('Average Weekly Revenue by Genre (Top 15)')
ax.invert_yaxis()
ax.bar_label(bars, fmt='$%.1fM', padding=4, fontsize=9)
plt.tight_layout()
plt.show()

print(df_genre[['genre', 'game_count', 'avg_weekly_revenue', 'median_weekly_revenue']]
      .to_string(index=False))

### Findings — Genre Revenue

**FPS ($81M) and RPG ($79.6M) lead** — both are genres with strong franchise loyalties and high-budget productions that command premium pricing and large audiences.

**The average vs median gap is significant.** For FPS, the average ($81M) is nearly 50% higher than the median ($55M). This tells us the distribution is right-skewed — a small number of blockbuster titles are pulling the average up. The *typical* FPS game earns closer to $55M per week, not $81M. In data, always check the median alongside the mean for skewed distributions.

**Indie sits at the bottom ($19.9M)** — consistent with real-world patterns where indie games serve niche audiences and price lower. Worth noting: the dataset labels some games as genre `"Indie"` while others use `is_indie = 1` as a boolean flag — these are not identical populations and their overlap is inconsistent in the simulation.

---

## Analysis 2 — Game Lifecycle

**Business question:** How do player counts and revenue change as a game ages through its lifecycle?

**Why this matters:** Understanding lifecycle phases helps publishers decide *when* to discount, when to launch DLC, and when a game has reached the end of its commercial life.

**Lifecycle phases in this dataset:**

| Phase | Period |
|---|---|
| `launch_week` | First 7 days after release |
| `launch_month` | Weeks 2–4 |
| `launch_quarter` | Months 2–3 |
| `first_year` | Months 4–12 |
| `second_year` | Year 2 |
| `long_tail` | Year 3 and beyond |

**How the query works:**
- We group all 5.58M rows by their `launch_phase` label
- For each phase we compute average players, revenue, hype score, and discount depth
- `WHERE launch_phase IS NOT NULL` excludes any rows where the phase value is missing

In [ ]:
df_lifecycle = con.execute("""
    SELECT
        launch_phase,
        ROUND(AVG(f.concurrent_players), 0)    AS avg_players,
        ROUND(AVG(f.estimated_revenue_usd), 2) AS avg_revenue,
        ROUND(AVG(f.hype_score), 3)            AS avg_hype,
        ROUND(AVG(f.discount_pct), 3)          AS avg_discount,
        COUNT(*)                               AS observation_count
    FROM fact_weekly_metrics f
    WHERE launch_phase IS NOT NULL
    GROUP BY launch_phase
    ORDER BY avg_players DESC
""").df()

# Order phases chronologically for the chart
phase_order = [
    'launch_week', 'launch_month', 'launch_quarter',
    'first_year', 'second_year', 'long_tail'
]
df_lifecycle['launch_phase'] = pd.Categorical(
    df_lifecycle['launch_phase'], categories=phase_order, ordered=True
)
df_lifecycle = df_lifecycle.sort_values('launch_phase')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].bar(
    df_lifecycle['launch_phase'],
    df_lifecycle['avg_players'] / 1_000,
    color='steelblue', edgecolor='white'
)
axes[0].set_title('Avg Concurrent Players by Lifecycle Phase')
axes[0].set_ylabel('Players (thousands)')
axes[0].tick_params(axis='x', rotation=30)

axes[1].bar(
    df_lifecycle['launch_phase'],
    df_lifecycle['avg_revenue'] / 1_000_000,
    color='coral', edgecolor='white'
)
axes[1].set_title('Avg Weekly Revenue by Lifecycle Phase')
axes[1].set_ylabel('Revenue (USD millions)')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

print(df_lifecycle.to_string(index=False))

### Findings — Game Lifecycle

**Players decay steadily** — from 110k at launch week down to 17k in the long tail. Expected: games lose players over time as novelty fades and newer releases compete for attention.

**Revenue tells the opposite story** — it *increases* from $10M at launch to $58.9M in the long tail. This seems backwards until you look at `avg_discount`:

| Phase | Avg players | Avg discount |
|---|---|---|
| launch_week | 110,955 | 0.45% |
| long_tail | 17,258 | 7.74% |

**The key insight:** Deep discounting in the long tail drives far more *purchases* per week, even with far fewer concurrent players. A game at $59 might sell 10,000 copies a week. The same game at $9.99 two years later might sell 200,000. This is the **discount-driven long tail revival** — a well-documented real-world phenomenon in digital storefronts like Steam's seasonal sales.

> **Interview talking point:** This is a good example of why you always look at multiple metrics together. Looking at player counts alone would suggest the long tail phase is commercially dead. Looking at revenue tells the opposite story.

---

## Analysis 3 — Platform Comparison

**Business question:** Which platform retains the most players and generates the most revenue?

**Why this matters:** Platform selection is a major strategic decision for publishers. A game on PC (Steam) behaves very differently from one on Nintendo Switch — different pricing norms, different audience demographics, different discount frequency.

**How the query works:**
- We `JOIN fact_weekly_metrics` to `dim_platform` on `platform_code`
- Each of the six platforms gets its own aggregated summary row
- `COUNT(DISTINCT f.game_id)` tells us how many unique games appear on each platform — a useful check that no platform was underrepresented in the simulation

In [ ]:
df_platform = con.execute("""
    SELECT
        p.platform,
        p.platform_code,
        ROUND(AVG(f.concurrent_players), 0)    AS avg_players,
        ROUND(AVG(f.estimated_revenue_usd), 2) AS avg_revenue,
        ROUND(AVG(f.hype_score), 3)            AS avg_hype,
        ROUND(AVG(f.discount_pct), 3)          AS avg_discount,
        ROUND(AVG(f.steam_rating_pct), 3)      AS avg_rating,
        COUNT(DISTINCT f.game_id)              AS unique_games
    FROM fact_weekly_metrics f
    JOIN dim_platform p ON f.platform_code = p.platform_code
    GROUP BY p.platform, p.platform_code
    ORDER BY avg_players DESC
""").df()

colors = ['#4e79a7', '#f28e2b', '#e15759', '#76b7b2', '#59a14f', '#edc948']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].bar(
    df_platform['platform'],
    df_platform['avg_players'] / 1_000,
    color=colors, edgecolor='white'
)
axes[0].set_title('Avg Concurrent Players by Platform')
axes[0].set_ylabel('Players (thousands)')
axes[0].tick_params(axis='x', rotation=25)

axes[1].bar(
    df_platform['platform'],
    df_platform['avg_revenue'] / 1_000_000,
    color=colors, edgecolor='white'
)
axes[1].set_title('Avg Weekly Revenue by Platform')
axes[1].set_ylabel('Revenue (USD millions)')
axes[1].tick_params(axis='x', rotation=25)

plt.tight_layout()
plt.show()

print(df_platform[[
    'platform', 'avg_players', 'avg_revenue',
    'avg_rating', 'avg_discount', 'unique_games'
]].to_string(index=False))

### Findings — Platform Comparison

**All six platforms perform remarkably similarly** — PlayStation 5 leads with 35k avg players and $61M avg revenue, but the gap to last place (Xbox One, 33k / $50M) is small. Ratings are essentially identical across all platforms (0.568–0.572).

**This is a simulation artifact.** In a real-world dataset you would expect much larger gaps:
- PC (Steam) typically dominates concurrent player counts with tens of millions of daily active users
- Nintendo Switch tends to have lower raw numbers but higher engagement-per-player and different pricing norms
- Legacy platforms (PS4, Xbox One) would show declining numbers as their install bases shrink

The simulation distributed games evenly across platforms with similar statistical properties, smoothing out the real-world differences that would make this analysis more interesting.

> **Data quality note:** Platform differentials are artificially uniform in this simulation. These numbers should not be used to draw real-world conclusions about platform performance.

---

## Analysis 4 — Discount Effectiveness

**Business question:** Do discounts actually drive player count increases?

**Why this matters:** Discounting is the most common lever publishers use to revive interest in older titles. But discounting also reduces revenue per unit — so the question is whether the volume uplift justifies the price reduction.

**How the query works:**
- We bucket `discount_pct` into six ranges using a `CASE WHEN` statement — SQL's equivalent of `pd.cut()` in pandas
- For each bucket we compute average players, revenue, and player growth rate
- `ORDER BY MIN(discount_pct)` ensures the buckets appear in logical order (low discount to high)

In [ ]:
df_discount = con.execute("""
    SELECT
        CASE
            WHEN discount_pct = 0                        THEN '0% (no discount)'
            WHEN discount_pct > 0  AND discount_pct < 10 THEN '1-9%'
            WHEN discount_pct >= 10 AND discount_pct < 25 THEN '10-24%'
            WHEN discount_pct >= 25 AND discount_pct < 50 THEN '25-49%'
            WHEN discount_pct >= 50 AND discount_pct < 75 THEN '50-74%'
            ELSE '75%+'
        END AS discount_bucket,
        ROUND(AVG(concurrent_players), 0)    AS avg_players,
        ROUND(AVG(estimated_revenue_usd), 2) AS avg_revenue,
        ROUND(AVG(player_growth_rate), 4)    AS avg_player_growth,
        COUNT(*)                             AS observation_count
    FROM fact_weekly_metrics
    GROUP BY discount_bucket
    ORDER BY MIN(discount_pct)
""").df()

colors_disc = ['#d3d3d3', '#b3cde3', '#8c96c6', '#8856a7', '#810f7c', '#4d004b']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].bar(
    df_discount['discount_bucket'],
    df_discount['avg_players'] / 1_000,
    color=colors_disc, edgecolor='white'
)
axes[0].set_title('Avg Players by Discount Level')
axes[0].set_ylabel('Players (thousands)')
axes[0].tick_params(axis='x', rotation=20)

axes[1].bar(
    df_discount['discount_bucket'],
    df_discount['avg_revenue'] / 1_000_000,
    color=colors_disc, edgecolor='white'
)
axes[1].set_title('Avg Weekly Revenue by Discount Level')
axes[1].set_ylabel('Revenue (USD millions)')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

print(df_discount.to_string(index=False))

### Findings — Discount Effectiveness

**The 1–9% bucket has the highest players (91k) — but this is misleading.** Small discounts tend to coincide with the launch phase, when organic player counts are naturally at their peak. The discount didn't cause the high player count — the lifecycle timing did. This is a classic **correlation vs causation** problem that is easy to miss when looking at aggregated data.

**Heavier discounts correlate with fewer players** — the 75%+ bucket has only 16.5k average players. Games only receive extreme discounts when they're old and have already lost most of their audience. The discount is a response to low engagement, not the cause of it.

**Revenue drops with discount depth** — from $59.9M at 0% discount to $13M at 75%+. Again, this reflects lifecycle position more than discount effectiveness in isolation.

**Data quality flag — `avg_player_growth` shows `inf`** — this is a division-by-zero problem. When a game had 0 players in a previous observation and then gained any players, the growth rate formula divides by zero, producing infinity. This column would need to be cleaned (e.g. replacing `inf` with `NaN`) before being used in any model or further analysis.

---

## Summary

| Question | Key finding |
|---|---|
| Which genres earn most? | FPS ($81M avg/week) and RPG ($79.6M) lead; Indie ($19.9M) trails. Average is skewed by blockbusters — median is more representative. |
| How does lifecycle affect metrics? | Players decay over time (110k → 17k) but revenue *increases* ($10M → $58.9M) driven by deepening discounts in later phases. |
| Which platform is best? | All six platforms perform nearly identically — an artifact of the simulation, not a real-world finding. |
| Do discounts drive player spikes? | Apparent correlation exists but causation is unclear — discount buckets overlap heavily with lifecycle phases. |

### Data quality notes
- `is_indie` flag disagrees with `genre = "Indie"` label for some games — two different systems of classification that aren't consistent
- `player_growth_rate` contains `inf` values from division by zero — needs cleaning before any modelling use
- Platform metrics are artificially uniform — the simulation did not model real platform-level behavioural differences

### What comes next
The `modeling/breakout_classifier.ipynb` notebook builds on this context to train a machine learning model that predicts which games become breakout hits — and uncovers a feature leakage issue in the process.